# Fase 3: Núcleo algorítmico y eficiencia

## Proyecto SIMCE 4.º Básico 2025 – Matemática

### 1. Introducción y objetivo de la fase

La Fase 3 tiene como propósito desarrollar y evaluar el núcleo algorítmico del proyecto, reutilizando el dataset procesado y validado durante la Fase 2.

En esta etapa se busca mejorar la organización del código mediante funciones reutilizables, comparar alternativas de implementación y evaluar su eficiencia a través de mediciones de tiempo de ejecución.

Asimismo, se preparará la evolución del proyecto hacia una arquitectura basada en programación orientada a objetos, considerando los requisitos de las evaluaciones formativa y sumativa.

El desarrollo de esta fase no modificará el dataset original ni reemplazará el proceso de preprocesamiento implementado en F2.

In [1]:
from pathlib import Path
import pandas as pd

# Identificar la carpeta principal del proyecto
RAIZ = Path.cwd()

if not (RAIZ / "data" / "processed").exists():
    RAIZ = RAIZ.parent

# Buscar el dataset final generado en F2
archivos = sorted(
    (RAIZ / "data" / "processed").glob(
        "simce4b2025_matematica_efectiva_*.csv"
    )
)

if not archivos:
    raise FileNotFoundError("No se encontró el dataset procesado de F2.")

# Cargar el dataset
ARCHIVO_F2 = archivos[-1]
df = pd.read_csv(ARCHIVO_F2)

print("Archivo:", ARCHIVO_F2.name)
print("Dimensiones:", df.shape)

Archivo: simce4b2025_matematica_efectiva_202609150013.csv
Dimensiones: (6524, 30)


In [2]:
# Mostrar las variables disponibles en el dataset

print("Columnas disponibles:")

for columna in df.columns:
    print("-", columna)

Columnas disponibles:
- rbd
- dv_rbd
- nombre_establecimiento
- asignatura
- cod_reg_rbd
- region
- cod_pro_rbd
- provincia
- cod_com_rbd
- comuna
- deprov
- pais
- ubicacion_region
- ubicacion_provincia
- ubicacion_comuna
- Zona
- Macrozona
- Orden
- dependencia_6_cat
- dependencia_4_cat
- grupo_socioeconomico
- ruralidad
- n_alumnos
- puntaje_promedio
- efectividad
- no_aplica
- codigo_bbdd
- fecha_bbdd
- grado
- anio


In [3]:
# Visualizar las regiones y sus puntajes

df[["region", "puntaje_promedio"]].head(10)

,region,puntaje_promedio
0,Los Lagos,263.0
1,Ñuble,221.0
2,Metropolitana de Santiago,220.0
3,Maule,243.0
4,Valparaíso,246.0
5,Metropolitana de Santiago,234.0
6,Metropolitana de Santiago,264.0
7,Maule,283.0
8,Metropolitana de Santiago,300.0
9,Coquimbo,281.0


In [4]:
# Calcular el promedio de Matemática por región

promedio_region_pandas = (
    df.groupby("region")["puntaje_promedio"]
    .mean()
    .reset_index()
)

# Mostrar los resultados
promedio_region_pandas

,region,puntaje_promedio
0,Antofagasta,256.358209
1,Arica y Parinacota,258.000000
2,Atacama,248.343137
3,Aysén del General Carlos Ibáñez del Campo,250.403846
4,Biobío,256.874150
5,Coquimbo,254.416667
6,La Araucanía,246.208084
7,Libertador General Bernardo O'Higgins,255.358670
8,Los Lagos,252.469636
9,Los Ríos,247.051793


In [5]:
# Calcular el promedio regional mediante un bucle

def promedio_por_region_bucle(datos):

    sumas = {}
    cantidades = {}

    for region, puntaje in zip(
        datos["region"],
        datos["puntaje_promedio"]
    ):

        if region not in sumas:
            sumas[region] = 0
            cantidades[region] = 0

        sumas[region] += puntaje
        cantidades[region] += 1

    resultado = []

    for region in sorted(sumas):

        promedio = sumas[region] / cantidades[region]

        resultado.append({
            "region": region,
            "puntaje_promedio": promedio
        })

    return pd.DataFrame(resultado)


# Ejecutar nuestra función
promedio_region_bucle = promedio_por_region_bucle(df)

promedio_region_bucle

,region,puntaje_promedio
0,Antofagasta,256.358209
1,Arica y Parinacota,258.000000
2,Atacama,248.343137
3,Aysén del General Carlos Ibáñez del Campo,250.403846
4,Biobío,256.874150
5,Coquimbo,254.416667
6,La Araucanía,246.208084
7,Libertador General Bernardo O'Higgins,255.358670
8,Los Lagos,252.469636
9,Los Ríos,247.051793


In [6]:
import numpy as np

# Comprobar que ambos métodos incluyen las mismas regiones
assert (
    promedio_region_pandas["region"].tolist()
    == promedio_region_bucle["region"].tolist()
), "Las regiones no coinciden."

# Comprobar que los promedios son equivalentes
np.testing.assert_allclose(
    promedio_region_pandas["puntaje_promedio"],
    promedio_region_bucle["puntaje_promedio"],
    rtol=0,
    atol=1e-10
)

print("✓ Ambos algoritmos producen resultados equivalentes.")
print("Regiones verificadas:", len(promedio_region_pandas))

✓ Ambos algoritmos producen resultados equivalentes.
Regiones verificadas: 16


In [7]:
import timeit

# Medir el método de Pandas
tiempos_pandas = timeit.repeat(
    lambda: df.groupby("region")["puntaje_promedio"].mean().reset_index(),
    repeat=5,
    number=10
)

# Medir nuestro algoritmo con for
tiempos_bucle = timeit.repeat(
    lambda: promedio_por_region_bucle(df),
    repeat=5,
    number=10
)

# Calcular el mejor tiempo promedio por ejecución
tiempo_pandas = min(tiempos_pandas) / 10 * 1000
tiempo_bucle = min(tiempos_bucle) / 10 * 1000

print(f"Tiempo Pandas: {tiempo_pandas:.3f} ms")
print(f"Tiempo bucle:  {tiempo_bucle:.3f} ms")

Tiempo Pandas: 2.163 ms
Tiempo bucle:  4.935 ms


### 2. Comparación de eficiencia algorítmica

Se implementaron dos alternativas para calcular el puntaje promedio de Matemática por región: una mediante la función `groupby()` de Pandas y otra mediante una función propia basada en un bucle `for`.

Ambas implementaciones fueron validadas y produjeron resultados numéricamente equivalentes para las 16 regiones del dataset.

Para comparar su eficiencia, se realizaron cinco rondas de medición con diez ejecuciones por ronda, seleccionando el menor tiempo promedio por ejecución.

Los resultados obtenidos fueron:

| Implementación | Tiempo de ejecución |
|---|---:|
| Pandas (`groupby`) | 2,246 ms |
| Bucle `for` | 4,951 ms |

En esta medición, Pandas presentó un menor tiempo de ejecución, siendo aproximadamente 2,2 veces más rápido que la implementación mediante un bucle.

Los resultados corresponden al entorno y dataset utilizados en esta ejecución, por lo que no constituyen una conclusión universal sobre el rendimiento de ambas alternativas.

## 3. Modularidad y reutilización del código

En esta sección se organiza el cálculo de los promedios regionales mediante funciones con responsabilidades específicas.

El propósito es facilitar la reutilización del código, evitar la repetición innecesaria de instrucciones y permitir que las implementaciones puedan ejecutarse y compararse de manera independiente.

Se utilizarán las dos alternativas desarrolladas anteriormente: el cálculo mediante Pandas y el algoritmo basado en un bucle `for`.

In [8]:
# Función reutilizable para calcular el promedio regional con Pandas

def promedio_por_region_pandas(datos):

    resultado = (
        datos.groupby("region")["puntaje_promedio"]
        .mean()
        .reset_index()
    )

    return resultado


# Ejecutar la función
resultado_pandas_modular = promedio_por_region_pandas(df)

resultado_pandas_modular

,region,puntaje_promedio
0,Antofagasta,256.358209
1,Arica y Parinacota,258.000000
2,Atacama,248.343137
3,Aysén del General Carlos Ibáñez del Campo,250.403846
4,Biobío,256.874150
5,Coquimbo,254.416667
6,La Araucanía,246.208084
7,Libertador General Bernardo O'Higgins,255.358670
8,Los Lagos,252.469636
9,Los Ríos,247.051793


In [9]:
import sys

# Permitir que Python encuentre los módulos del proyecto
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Importar nuestra función desde el archivo independiente
from src.indicadores_f3 import (
    promedio_por_region_pandas as promedio_pandas_modulo
)

# Ejecutar la función importada
resultado_modulo = promedio_pandas_modulo(df)

resultado_modulo.head()

,region,puntaje_promedio
0,Antofagasta,256.358209
1,Arica y Parinacota,258.000000
2,Atacama,248.343137
3,Aysén del General Carlos Ibáñez del Campo,250.403846
4,Biobío,256.874150


In [10]:
# Importar nuestro algoritmo con for desde el módulo

from src.indicadores_f3 import (
    promedio_por_region_bucle as promedio_bucle_modulo
)

# Ejecutar la función importada
resultado_bucle_modulo = promedio_bucle_modulo(df)

# Comparar los resultados de ambas funciones
pd.testing.assert_frame_equal(
    resultado_modulo,
    resultado_bucle_modulo,
    check_exact=False,
    atol=1e-10,
    rtol=0
)

print("✓ Ambas funciones del módulo producen resultados equivalentes.")
print("Regiones verificadas:", len(resultado_modulo))

✓ Ambas funciones del módulo producen resultados equivalentes.
Regiones verificadas: 16
